In [3]:
import numpy as np

np.random.seed(42)
N = 1000

# ── 1. Base loan portfolio matrix (rows = loans, cols = features) ──────────
#    Used for: pairwise distances, covariance matrix, upper triangle, normalise
X = np.column_stack([
    np.random.lognormal(13.0, 0.7, N),          # loan_amount
    np.random.beta(2, 3, N) * 1.4,              # LTV
    np.random.lognormal(12.5, 0.6, N),          # income
    np.random.exponential(36, N).clip(1, 240),  # months_active
    np.random.uniform(0, 1, N),                 # recovery_rate
])
feature_names = ['loan_amount', 'LTV', 'income', 'months_active', 'recovery_rate']

# ── 2. 1D time series array — monthly portfolio loss rates (%) ────────────
#    Used for: rolling mean (np.convolve), z-score, normalise
monthly_losses = np.clip(
    0.03
    + 0.01 * np.sin(np.linspace(0, 4 * np.pi, 60))   # cyclical macro pattern
    + np.random.normal(0, 0.005, 60),                  # noise
    0, None
)

# ── 3. Return series matrix (rows = months, cols = assets) ────────────────
#    Used for: covariance matrix, correlation, upper triangle extraction
returns = np.random.multivariate_normal(
    mean=[0.001, 0.002, 0.0015, 0.0005, 0.003],
    cov=np.array([
        [0.0004, 0.0002, 0.0001, 0.00005, 0.00015],
        [0.0002, 0.0009, 0.0003, 0.00010, 0.00020],
        [0.0001, 0.0003, 0.0006, 0.00008, 0.00012],
        [0.00005, 0.0001, 0.00008, 0.0003, 0.00006],
        [0.00015, 0.0002, 0.00012, 0.00006, 0.0008],
    ]),
    size=120   # 120 months = 10 years
)
asset_names = ['DK_Mortgage', 'Corp_Bonds', 'Equities', 'Gov_Bonds', 'EM_Debt']

# ── 4. Score array with outliers injected ────────────────────────────────
#    Used for: IQR outlier detection, clipping, condition counts
credit_scores = np.concatenate([
    np.random.normal(650, 80, N - 30),   # normal borrowers
    np.random.uniform(200, 300, 15),     # very bad — injected outliers
    np.random.uniform(850, 1000, 15),    # suspiciously perfect — injected outliers
])
np.random.shuffle(credit_scores)

# ── 5. Boolean / threshold arrays ────────────────────────────────────────
#    Used for: (arr > threshold).sum(), np.where oneliners
LTV_arr      = X[:, 1]
threshold    = 0.8
is_high_risk = LTV_arr > threshold

# ── 6. 2D distance demo matrix (smaller, for readable output) ────────────
#    Used for: pairwise Euclidean distance oneliner
X_small = X[:10, [1, 4]]   # 10 loans, LTV + recovery_rate only

# ── Summary ───────────────────────────────────────────────────────────────
print("=== Oneliners_3 Data Summary ===")
print(f"\nX  (loan matrix)      : {X.shape}  — {feature_names}")
print(f"monthly_losses        : {monthly_losses.shape}  — 60 months of portfolio loss rates")
print(f"returns               : {returns.shape}  — {asset_names}")
print(f"credit_scores         : {credit_scores.shape}  — with {(credit_scores < 350).sum()} low + {(credit_scores > 850).sum()} high outliers injected")
print(f"is_high_risk (LTV>0.8): {is_high_risk.sum()} loans ({is_high_risk.mean()*100:.1f}%)")
print(f"X_small               : {X_small.shape}  — for pairwise distance demo")

print("\n--- monthly_losses (first 12 months) ---")
print(np.round(monthly_losses[:12], 5))

print("\n--- X feature stats ---")
for i, name in enumerate(feature_names):
    print(f"  {name:<16}  mean={X[:,i].mean():.3f}  std={X[:,i].std():.3f}  min={X[:,i].min():.3f}  max={X[:,i].max():.3f}")

print("\n--- Returns correlation matrix ---")
print(np.round(np.corrcoef(returns.T), 3))

=== Oneliners_3 Data Summary ===

X  (loan matrix)      : (1000, 5)  — ['loan_amount', 'LTV', 'income', 'months_active', 'recovery_rate']
monthly_losses        : (60,)  — 60 months of portfolio loss rates
returns               : (120, 5)  — ['DK_Mortgage', 'Corp_Bonds', 'Equities', 'Gov_Bonds', 'EM_Debt']
credit_scores         : (1000,)  — with 15 low + 25 high outliers injected
is_high_risk (LTV>0.8): 220 loans (22.0%)
X_small               : (10, 2)  — for pairwise distance demo

--- monthly_losses (first 12 months) ---
[0.02801 0.03616 0.03316 0.04227 0.04076 0.04077 0.039   0.04156 0.04135
 0.0367  0.0346  0.03218]

--- X feature stats ---
  loan_amount       mean=571059.785  std=472659.357  min=45757.586  max=6562695.943
  LTV               mean=0.566  std=0.279  min=0.012  max=1.311
  income            mean=309865.350  std=195089.085  min=31280.883  max=1637738.731
  months_active     mean=33.935  std=33.916  min=1.000  max=240.000
  recovery_rate     mean=0.486  std=0.279  min=0

In [5]:
# arr        → monthly_losses  (1D array)
# X          → X               (2D matrix)
# threshold  → threshold       (already defined as 0.8)

# Normalise to [0, 1]
monthly_losses_norm = (monthly_losses - monthly_losses.min()) / (monthly_losses.max() - monthly_losses.min())

# Z-score standardise
monthly_losses_z = (monthly_losses - monthly_losses.mean()) / monthly_losses.std()

# Rolling 7-month mean (smoothing the macro cycle)
rolling_mean = np.convolve(monthly_losses, np.ones(7) / 7, mode='valid')

# Pairwise Euclidean distances between 10 loans (LTV + recovery_rate)
distances = np.sqrt(((X_small[:, None] - X_small[None, :]) ** 2).sum(-1))

# Count loans above LTV threshold
(LTV_arr > threshold).sum()

# Covariance matrix of asset returns, upper triangle only
np.triu(np.cov(returns.T), k=1)

# Normalise each column of X independently (column-wise min-max)
X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

# Z-score each column of X independently
X_z = (X - X.mean(axis=0)) / X.std(axis=0)

# Count outliers in credit_scores using IQR
q1, q3 = np.percentile(credit_scores, [25, 75])
(credit_scores < q1 - 1.5*(q3-q1)).sum() + (credit_scores > q3 + 1.5*(q3-q1)).sum()

# Clip credit_scores to IQR bounds (Winsorise)
credit_scores_clipped = np.clip(credit_scores, q1 - 1.5*(q3-q1), q3 + 1.5*(q3-q1))